# RAG Corpus Enrichment

Joins KEV and EPSS onto `data/rag_corpus.jsonl` (12,000 records).

Adds three fields per record:
- `kev_listed` (bool): whether the CVE appears in the CISA Known Exploited Vulnerabilities catalogue
- `epss_score` (float | null): FIRST EPSS probability score (0-1)
- `epss_percentile` (float | null): EPSS percentile rank (0-1)

Output: `data/rag_corpus_enriched.jsonl`

In [1]:
import json
import csv
import statistics
from collections import Counter
from pathlib import Path

DATA_DIR = Path("../data")

## Load RAG corpus

In [2]:
corpus = []
with open(DATA_DIR / "rag_corpus.jsonl") as f:
    for line in f:
        corpus.append(json.loads(line))

print(f"Loaded {len(corpus):,} records")
print(f"Fields: {list(corpus[0].keys())}")

Loaded 12,000 records
Fields: ['id', 'published', 'lastModified', 'description', 'cvss_score', 'cvss_severity', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'confidentiality_impact', 'integrity_impact', 'availability_impact', 'configurations', 'references']


## Load KEV catalogue

In [3]:
kev_ids = set()
with open(DATA_DIR / "known_exploited_vulnerabilities.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        kev_ids.add(row["cveID"].strip())

print(f"KEV catalogue: {len(kev_ids):,} entries")

KEV catalogue: 1,623 entries


## Load EPSS scores

In [5]:
def _float_or_none(val):
    return float(val) if val and val.strip() else None

epss_lookup = {}
with open(DATA_DIR / "epss_scores.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        epss_lookup[row["cve_id"].strip()] = {
            "epss_score": _float_or_none(row["epss"]),
            "epss_percentile": _float_or_none(row["percentile"]),
        }

print(f"EPSS scores: {len(epss_lookup):,} entries")
empty = sum(1 for v in epss_lookup.values() if v["epss_score"] is None)
print(f"Entries with no score: {empty:,}")

EPSS scores: 156,084 entries
Entries with no score: 11


## Enrich corpus

In [6]:
enriched = []
for record in corpus:
    cve_id = record["id"]
    enriched_record = dict(record)
    enriched_record["kev_listed"] = cve_id in kev_ids
    epss = epss_lookup.get(cve_id)
    enriched_record["epss_score"] = epss["epss_score"] if epss else None
    enriched_record["epss_percentile"] = epss["epss_percentile"] if epss else None
    enriched.append(enriched_record)

print(f"Enriched: {len(enriched):,} records")
print(f"New fields: {[k for k in enriched[0] if k not in corpus[0]]}")

Enriched: 12,000 records
New fields: ['kev_listed', 'epss_score', 'epss_percentile']


## Coverage summary

In [7]:
kev_count = sum(1 for r in enriched if r["kev_listed"])
epss_missing = sum(1 for r in enriched if r["epss_score"] is None)
epss_scores = [r["epss_score"] for r in enriched if r["epss_score"] is not None]

print(f"KEV-listed:       {kev_count:>6,}  ({kev_count / len(enriched) * 100:.1f}%)")
print(f"EPSS coverage:    {len(epss_scores):>6,}  ({len(epss_scores) / len(enriched) * 100:.1f}%)")
print(f"EPSS missing:     {epss_missing:>6,}")
if epss_scores:
    print(f"EPSS mean:        {statistics.mean(epss_scores):.4f}")
    print(f"EPSS median:      {statistics.median(epss_scores):.4f}")
    print(f"EPSS >= 0.5:      {sum(1 for s in epss_scores if s >= 0.5):>6,}")

KEV-listed:           65  (0.5%)
EPSS coverage:    11,999  (100.0%)
EPSS missing:          1
EPSS mean:        0.0198
EPSS median:      0.0049
EPSS >= 0.5:         128


## KEV and EPSS breakdown by severity

In [8]:
sev_counts = Counter(r["cvss_severity"] for r in enriched)
kev_by_sev = Counter(r["cvss_severity"] for r in enriched if r["kev_listed"])
epss_by_sev = {
    sev: [r["epss_score"] for r in enriched if r["cvss_severity"] == sev and r["epss_score"] is not None]
    for sev in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]
}

print(f"{'Severity':<10} {'Total':>7} {'KEV':>6} {'KEV%':>6} {'EPSS mean':>10}")
print("-" * 44)
for sev in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
    total = sev_counts.get(sev, 0)
    kev = kev_by_sev.get(sev, 0)
    kev_pct = kev / total * 100 if total else 0
    scores = epss_by_sev[sev]
    epss_mean = f"{statistics.mean(scores):.4f}" if scores else "n/a"
    print(f"{sev:<10} {total:>7,} {kev:>6,} {kev_pct:>5.1f}% {epss_mean:>10}")

Severity     Total    KEV   KEV%  EPSS mean
--------------------------------------------
CRITICAL     1,355     29   2.1%     0.0679
HIGH         4,413     23   0.5%     0.0193
MEDIUM       5,720     13   0.2%     0.0101
LOW            512      0   0.0%     0.0048


## Save enriched corpus

In [9]:
out_path = DATA_DIR / "rag_corpus_enriched.jsonl"
with open(out_path, "w") as f:
    for record in enriched:
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(enriched):,} records -> {out_path}")

# Spot-check a KEV-listed record
sample = next((r for r in enriched if r["kev_listed"]), None)
if sample:
    print(f"\nSample KEV record: {sample['id']}")
    print(f"  kev_listed:       {sample['kev_listed']}")
    print(f"  epss_score:       {sample['epss_score']}")
    print(f"  epss_percentile:  {sample['epss_percentile']}")
    print(f"  cvss_severity:    {sample['cvss_severity']}")

Saved 12,000 records -> ../data/rag_corpus_enriched.jsonl

Sample KEV record: CVE-2021-26084
  kev_listed:       True
  epss_score:       0.99999
  epss_percentile:  0.99992
  cvss_severity:    CRITICAL
